Road network validation

In [1]:
from google.colab import drive
import os

# Montar el drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
import rasterio
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# 1. Configuración de Rutas Absolutas
# Cambia 'Mi unidad' por 'MyDrive' si tu entorno está en inglés
base_path = "/content/drive/MyDrive/DOCTORADO 2026/VISION ARTIFICIAL/E4"
input_dir = os.path.join(base_path, "Input data")
output_dir = os.path.join(base_path, "Road network validation")

# Verificación de existencia de carpeta de salida
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Paths específicos de archivos
path_cem = os.path.join(input_dir, "2. Mapa Veracruz georeferenciado.tif")
path_urbano = os.path.join(input_dir, "1. Capa urbana Veracruz.tif")
UMBRAL_CRITICO = 3.0

def ejecutar_validacion_vial():
    # Verificación de existencia de archivos antes de abrir
    if not os.path.exists(path_cem):
        print(f"ERROR: No se encontró el CEM en {path_cem}")
        return

    # 2. Carga de Datos
    with rasterio.open(path_cem) as dem_src:
        dem = dem_src.read(1)
        meta = dem_src.meta.copy()
        res = dem_src.res[0]

    with rasterio.open(path_urbano) as urban_src:
        # Alineación automática por si las dimensiones varían levemente
        urbano = urban_src.read(1, out_shape=dem.shape)

    # 3. Lógica de Validación Estructural
    mascara_vial = (urbano > 0)
    zona_inundable = (dem <= UMBRAL_CRITICO) & (dem > -999)
    vialidades_afectadas = mascara_vial & zona_inundable

    # 4. Cálculo de Métricas
    area_pixel = res * res
    total_vial_m2 = np.sum(mascara_vial) * area_pixel
    afectado_vial_m2 = np.sum(vialidades_afectadas) * area_pixel
    porcentaje_exposicion = (afectado_vial_m2 / total_vial_m2) * 100

    # 5. Guardar Resultados
    # A. GeoTIFF
    meta.update(dtype=rasterio.uint8, count=1)
    with rasterio.open(os.path.join(output_dir, "Red_Vial_Comprometida_3m.tif"), 'w', **meta) as dst:
        dst.write(vialidades_afectadas.astype(rasterio.uint8), 1)

    # B. Reporte CSV
    df_reporte = pd.DataFrame({
        "Metrica": ["Area Vial Total (m2)", "Area Vial Expuesta (m2)", "Porcentaje de Exposicion (%)"],
        "Valor": [total_vial_m2, afectado_vial_m2, porcentaje_exposicion]
    })
    df_reporte.to_csv(os.path.join(output_dir, "Reporte_Exposicion_Vial.csv"), index=False)

    # C. Imagen de Sustento (.png)
    plt.figure(figsize=(12, 10))
    plt.title(f"Validación de Red Vial - Veracruz\nUmbral: {UMBRAL_CRITICO} msnm")
    plt.imshow(dem, cmap='terrain', alpha=0.6)
    vial_plot = np.ma.masked_where(~vialidades_afectadas, vialidades_afectadas)
    plt.imshow(vial_plot, cmap='Reds', interpolation='none')
    plt.colorbar(label='Elevación (msnm)')

    textstr = f'Exposición Vial: {porcentaje_exposicion:.2f}%'
    plt.gcf().text(0.15, 0.15, textstr, fontsize=12, bbox=dict(facecolor='white', alpha=0.8))

    plt.savefig(os.path.join(output_dir, "Visualizacion_Validacion_Vial.png"), dpi=300)
    plt.close()

    print(f"Éxito. Archivos guardados en: {output_dir}")

if __name__ == "__main__":
    ejecutar_validacion_vial()

Éxito. Archivos guardados en: /content/drive/MyDrive/DOCTORADO 2026/VISION ARTIFICIAL/E4/Road network validation


SEGMENTATION

In [4]:
import os
import rasterio
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import ndimage

# 1. Configuración de Rutas (Paths Absolutos)
base_path = "/content/drive/MyDrive/DOCTORADO 2026/VISION ARTIFICIAL/E4"
input_dir = os.path.join(base_path, "Input data")
output_dir = os.path.join(base_path, "Settlement exposure segmentation")

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

path_cem = os.path.join(input_dir, "2. Mapa Veracruz georeferenciado.tif")
path_manzanas = os.path.join(input_dir, "1. Manzanas segmentadas Veracruz.png")
UMBRAL_CRITICO = 3.0

def ejecutar_segmentacion_asentamientos():
    # 2. Carga de Datos
    with rasterio.open(path_cem) as dem_src:
        dem = dem_src.read(1)
        meta = dem_src.meta.copy()

    # Cargar PNG y convertir a máscara binaria (ajustando dimensiones al CEM)
    import PIL.Image
    img_manzanas = PIL.Image.open(path_manzanas).convert('L')
    img_manzanas = img_manzanas.resize((dem.shape[1], dem.shape[0]), PIL.Image.NEAREST)
    manzanas_mask = np.array(img_manzanas) > 128 # Umbral para binarizar

    # 3. Identificación de Manzanas Individuales (Etiquetado)
    # Esto nos permite analizar cada manzana como un objeto independiente
    label_im, nb_labels = ndimage.label(manzanas_mask)

    # 4. Análisis de Riesgo por Objeto
    # Creamos un array vacío para la clasificación final
    clasificacion_map = np.zeros_like(dem, dtype=np.uint8)
    datos_resumen = []

    for i in range(1, nb_labels + 1):
        mask_obj = (label_im == i)
        elevaciones_manzana = dem[mask_obj]

        # Ignorar valores NoData del CEM
        elevaciones_validas = elevaciones_manzana[elevaciones_manzana > -999]

        if len(elevaciones_validas) > 0:
            media_elev = np.mean(elevaciones_validas)

            # Clasificación basada en la media de elevación de la manzana
            if media_elev <= (UMBRAL_CRITICO * 0.5):
                nivel = 3 # Alto Riesgo (Rojo)
                cat = "Alto"
            elif media_elev <= UMBRAL_CRITICO:
                nivel = 2 # Riesgo Medio (Naranja)
                cat = "Medio"
            else:
                nivel = 1 # Riesgo Bajo (Azul/Verde)
                cat = "Bajo"

            clasificacion_map[mask_obj] = nivel
            datos_resumen.append(cat)

    # 5. Generar Reporte y Archivos
    # A. CSV
    conteo = pd.Series(datos_resumen).value_counts().reset_index()
    conteo.columns = ['Nivel de Riesgo', 'Cantidad de Manzanas']
    conteo.to_csv(os.path.join(output_dir, "Estadisticas_Asentamientos.csv"), index=False)

    # B. GeoTIFF Clasificado
    meta.update(dtype=rasterio.uint8, count=1)
    with rasterio.open(os.path.join(output_dir, "Clasificacion_Riesgo_Manzanas.tif"), 'w', **meta) as dst:
        dst.write(clasificacion_map, 1)

    # C. PNG de Sustento
    plt.figure(figsize=(12, 10))
    plt.title("Segmentación de Exposición por Manzanas - Veracruz", fontsize=14)
    plt.imshow(dem, cmap='gray', alpha=0.3) # Fondo topográfico tenue

    # Mapa de colores: 1:Bajo(Verde), 2:Medio(Naranja), 3:Alto(Rojo)
    from matplotlib.colors import ListedColormap
    cmap_custom = ListedColormap(['none', '#2ecc71', '#f39c12', '#e74c3c'])
    plt.imshow(clasificacion_map, cmap=cmap_custom, interpolation='none')

    plt.colorbar(ticks=[1, 2, 3], label='Nivel de Riesgo (1:Bajo, 2:Medio, 3:Alto)')
    plt.savefig(os.path.join(output_dir, "Mapa_Calor_Asentamientos.png"), dpi=300)
    plt.close()

    print(f"Paso 2 completado exitosamente en: {output_dir}")

if __name__ == "__main__":
    ejecutar_segmentacion_asentamientos()

Paso 2 completado exitosamente en: /content/drive/MyDrive/DOCTORADO 2026/VISION ARTIFICIAL/E4/Settlement exposure segmentation


Topographic consistency analysis

In [5]:
import os
import rasterio
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import ndimage

# 1. Configuración de Rutas
base_path = "/content/drive/MyDrive/DOCTORADO 2026/VISION ARTIFICIAL/E4"
input_dir = os.path.join(base_path, "Input data")
output_dir = os.path.join(base_path, "Topographic consistency analysis")

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

path_cem = os.path.join(input_dir, "2. Mapa Veracruz georeferenciado.tif")
path_urbano = os.path.join(input_dir, "1. Capa urbana Veracruz.tif")

def analizar_consistencia_topografica():
    # 2. Carga de Datos
    with rasterio.open(path_cem) as dem_src:
        dem = dem_src.read(1)
        meta = dem_src.meta.copy()

    with rasterio.open(path_urbano) as urban_src:
        urbano = urban_src.read(1, out_shape=dem.shape)
        mask_urbana = (urbano > 0)

    # 3. Cálculo de Gradientes (Detección de Bordes)
    # Buscamos cambios bruscos en la elevación que coincidan con el borde de la máscara
    grad_dem = np.gradient(dem)
    magnitud_grad = np.sqrt(grad_dem[0]**2 + grad_dem[1]**2)

    # Extraer bordes de la capa urbana usando un filtro de Sobel o gradiente morfológico
    bordes_urbanos = ndimage.binary_dilation(mask_urbana) ^ mask_urbana

    # 4. Identificación de Ruido en Bordes
    # El error ocurre si hay una pendiente extrema (>45°) justo en el borde de una manzana
    error_bordes = magnitud_grad * bordes_urbanos
    threshold_ruido = np.percentile(magnitud_grad, 95) # Umbral estadístico de ruido
    mapa_error = (error_bordes > threshold_ruido).astype(np.uint8)

    # 5. Generación de Métricas y Reportes
    elev_urbana = dem[mask_urbana]
    elev_no_urbana = dem[~mask_urbana]

    stats = {
        "Desviación Estándar Urbana": np.std(elev_urbana[elev_urbana > -999]),
        "Error Promedio en Bordes": np.mean(error_bordes[bordes_urbanos]),
        "Píxeles con Ruido Detectado": np.sum(mapa_error)
    }

    # A. Guardar CSV
    pd.DataFrame([stats]).to_csv(os.path.join(output_dir, "Metricas_Consistencia.csv"), index=False)

    # B. GeoTIFF de Error
    meta.update(dtype=rasterio.uint8, count=1)
    with rasterio.open(os.path.join(output_dir, "Mapa_Error_Bordes.tif"), 'w', **meta) as dst:
        dst.write(mapa_error, 1)

    # C. Gráfico de Dispersión (Elevación vs Densidad de Borde)
    plt.figure(figsize=(10, 6))
    plt.scatter(dem[bordes_urbanos], magnitud_grad[bordes_urbanos], alpha=0.1, s=1, c='blue')
    plt.axhline(y=threshold_ruido, color='r', linestyle='--', label='Umbral de Ruido')
    plt.title("Correlación Elevación vs. Gradiente en Bordes Urbanos")
    plt.xlabel("Elevación (msnm)")
    plt.ylabel("Magnitud del Gradiente (Pendiente)")
    plt.legend()
    plt.savefig(os.path.join(output_dir, "Scatter_Elevacion_Urbana.png"), dpi=300)
    plt.close()

    print(f"Paso 3 completado. Reporte de consistencia generado en: {output_dir}")

if __name__ == "__main__":
    analizar_consistencia_topografica()

/tmp/ipykernel_1960/227572158.py:69: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.savefig(os.path.join(output_dir, "Scatter_Elevacion_Urbana.png"), dpi=300)


Paso 3 completado. Reporte de consistencia generado en: /content/drive/MyDrive/DOCTORADO 2026/VISION ARTIFICIAL/E4/Topographic consistency analysis


Validation

In [6]:
import os
import json
import numpy as np
import pandas as pd
import rasterio

# 1. Configuración de Rutas
base_path = "/content/drive/MyDrive/DOCTORADO 2026/VISION ARTIFICIAL/E4"
output_dir = os.path.join(base_path, "PyTorch Metadata")
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Rutas de resultados previos para recolección
path_vial = os.path.join(base_path, "Road network validation/Reporte_Exposicion_Vial.csv")
path_asentamientos = os.path.join(base_path, "Settlement exposure segmentation/Estadisticas_Asentamientos.csv")
path_consistencia = os.path.join(base_path, "Topographic consistency analysis/Metricas_Consistencia.csv")
path_riesgo_map = os.path.join(base_path, "Settlement exposure segmentation/Clasificacion_Riesgo_Manzanas.tif")

def finalizar_validacion_estructural():
    # 2. Recolección de Datos Estadísticos
    try:
        df_vial = pd.read_csv(path_vial)
        df_asent = pd.read_csv(path_asentamientos)
        df_consist = pd.read_csv(path_consistencia)

        metadata = {
            "proyecto": "Modelo de inundaciones Veracruz",
            "etapa": "E4 - Validación Estructural",
            "metricas_viales": df_vial.set_index('Metrica')['Valor'].to_dict(),
            "distribucion_riesgo_asentamientos": df_asent.set_index('Nivel de Riesgo')['Cantidad de Manzanas'].to_dict(),
            "calidad_topografica": df_consist.iloc[0].to_dict()
        }
    except Exception as e:
        print(f"Error recolectando archivos: {e}")
        return

    # 3. Preparación del Ground Truth para PyTorch
    # Convertimos el mapa de riesgo de manzanas en un tensor binario/categórico
    with rasterio.open(path_riesgo_map) as src:
        ground_truth = src.read(1)
        # Guardar en formato NumPy para carga rápida en modelos de IA
        np.save(os.path.join(output_dir, "Ground_Truth_Tensor.npy"), ground_truth)

    # 4. Exportar Metadatos en JSON
    with open(os.path.join(output_dir, "Metadata_Dataset.json"), 'w') as f:
        json.dump(metadata, f, indent=4)

    # 5. Reporte Ejecutivo Final
    with open(os.path.join(output_dir, "Reporte_Final_E4.txt"), 'w') as f:
        f.write("RESUMEN DE VALIDACIÓN ESTRUCTURAL - ETAPA 4\n")
        f.write("==========================================\n\n")
        f.write(f"Exposición Vial Total: {metadata['metricas_viales'].get('Porcentaje de Exposicion (%)', 0):.2f}%\n")
        f.write(f"Píxeles con Ruido Topográfico: {metadata['calidad_topografica'].get('Píxeles con Ruido Detectado', 0)}\n")
        f.write("\nEstado de Validación: EXITOSO - Dataset listo para PyTorch.")

    print(f"E4 finalizada. Metadatos y Tensores disponibles en: {output_dir}")

if __name__ == "__main__":
    finalizar_validacion_estructural()

E4 finalizada. Metadatos y Tensores disponibles en: /content/drive/MyDrive/DOCTORADO 2026/VISION ARTIFICIAL/E4/PyTorch Metadata
